# Yearly pairwise agreement
The API performs aggregation and export; this notebook presents the persisted result.

In [ ]:
from datetime import date
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from liljegren_analysis.comparison import (
    STATISTIC_SPECS, analyze_yearly_pairwise, pairwise_matrices,
)
from liljegren_analysis.comparison.config import REFERENCE_IMPLEMENTATION

REPOSITORY = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_ROOT = REPOSITORY / 'data/pairwise/6-targets'
OUTPUT_ROOT = REPOSITORY / 'data/pairwise/yearly/2025'
START_DATE = date(2025, 1, 1)
END_DATE = date(2025, 12, 31)
SAMPLE_SCOPE = 'all-implementations'
REFERENCE = REFERENCE_IMPLEMENTATION
EXPORT_IMPLEMENTATIONS = (
    'heatstress', 'meteor', 'thermofeel',
    'kwodzicki-pywbgt', 'kong-huber-pywbgt',
)
IMPLEMENTATION_ORDER = (REFERENCE, *EXPORT_IMPLEMENTATIONS)
DEFAULT_QUANTITY = 'wbgt'
DEFAULT_STATISTIC = 'mean_difference_c'

In [ ]:
OUTPUTS = analyze_yearly_pairwise(
    INPUT_ROOT, start_date=START_DATE, end_date=END_DATE,
    output_root=OUTPUT_ROOT, sample_scope=SAMPLE_SCOPE,
    reference_implementation=REFERENCE,
    export_implementations=EXPORT_IMPLEMENTATIONS,
)
YEARLY = pd.read_parquet(OUTPUTS.pairwise)
COVERAGE = pd.read_parquet(OUTPUTS.quantity_coverage)
MATRICES = pairwise_matrices(YEARLY)
display(COVERAGE)
OUTPUTS

## Interactive comparison matrix

In [ ]:
def current_matrix(quantity, statistic):
    return MATRICES[quantity][statistic].reindex(
        index=IMPLEMENTATION_ORDER, columns=IMPLEMENTATION_ORDER,
    )


def render_matrix(_=None):
    quantity = QUANTITY_CONTROL.value
    statistic = STATISTIC_CONTROL.value
    matrix = current_matrix(quantity, statistic)
    signed = STATISTIC_SPECS[statistic]['signed']
    figure = go.Figure(go.Heatmap(
        z=matrix.astype(object).where(matrix.notna(), None).to_numpy().tolist(),
        x=matrix.columns.tolist(), y=matrix.index.tolist(),
        text=matrix.map(
            lambda value: '' if pd.isna(value) else f'{value:,.2f}'
        ).to_numpy().tolist(),
        texttemplate='%{text}', colorscale='RdBu_r' if signed else 'YlOrRd',
        zmid=0 if signed else None,
        hovertemplate='<b>%{y} − %{x}</b><br>%{z:,.3f}<extra></extra>',
    ))
    figure.update_layout(
        template='plotly_dark', title=f'{quantity.upper()}: {statistic}', height=760,
        xaxis={'title': 'implementation_b'},
        yaxis={'title': 'implementation_a', 'autorange': 'reversed', 'scaleanchor': 'x'},
    )
    with MATRIX_OUTPUT:
        MATRIX_OUTPUT.clear_output(wait=False)
        display(figure)


def export_current_matrix(_):
    quantity = QUANTITY_CONTROL.value
    statistic = STATISTIC_CONTROL.value
    matrix = current_matrix(quantity, statistic)
    output = OUTPUT_ROOT / f'pairwise_matrix_{quantity}_{statistic}.csv'
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    matrix.rename_axis('implementation_a').to_csv(output)
    EXPORT_STATUS.value = str(output)


QUANTITY_CONTROL = widgets.Dropdown(options=tuple(MATRICES), value=DEFAULT_QUANTITY, description='Quantity')
STATISTIC_CONTROL = widgets.Dropdown(options=tuple(STATISTIC_SPECS), value=DEFAULT_STATISTIC, description='Statistic')
EXPORT_BUTTON = widgets.Button(description='Export CSV', icon='download', button_style='primary')
EXPORT_STATUS = widgets.Label()
MATRIX_OUTPUT = widgets.Output()
QUANTITY_CONTROL.observe(render_matrix, names='value')
STATISTIC_CONTROL.observe(render_matrix, names='value')
EXPORT_BUTTON.on_click(export_current_matrix)
display(widgets.HBox([QUANTITY_CONTROL, STATISTIC_CONTROL, EXPORT_BUTTON]))
display(EXPORT_STATUS)
display(MATRIX_OUTPUT)
render_matrix()